# Home Credit Default Risk — 探索性資料分析 (EDA)

本 notebook 記錄信用風險違約預測專題的資料探索過程，涵蓋：資料規模與目標變數分佈、缺失值型態分析、
異常值偵測與驗證、金融比率特徵建構與相關性檢查、以及附屬表格（`bureau.csv`、`previous_application.csv`）
的聚合與驗證。每一步的處理邏輯都先在這裡驗證過，確認合理之後才收斂進 `src/features.py` 與
`src/build_features.py` 的正式 pipeline，供 train/test 共用。

## 1. 資料載入與記憶體優化

原始 `application_train.csv` 讀入後，數值欄位預設會用 `int64`/`float64` 儲存，但實際數值範圍往往用不到
這麼大的型態（例如 0/1 的旗標欄位）。這裡使用自訂的 `reduce_mem_usage()` 函式，逐欄檢查數值範圍後
downcast 成最小足夠的型態，並將文字欄位轉為 `category`，藉此大幅降低記憶體用量，這在 8GB 記憶體的機器上
處理後續多張百萬列表格時是必要的前提工程。

In [1]:
import sys
sys.path.append('..')  # 讓 Python 找得到 src/ 底下的模組

import pandas as pd
import numpy as np
from src.utils import reduce_mem_usage

In [2]:
app_train = pd.read_csv('../data/application_train.csv')
app_train = reduce_mem_usage(app_train)

print(app_train.shape)

記憶體用量：325.22 MB → 167.15 MB (降低 48.6%)
(307511, 122)


## 2. 目標變數分佈

`TARGET` 是本專題要預測的標籤：1 代表違約、0 代表正常還款。信用風控資料集的典型特徵是類別極度不平衡，
這會直接影響後續模型訓練時的策略選擇（例如 `scale_pos_weight`、`class_weight='balanced'`）與評估指標的
選擇（不能只看 Accuracy）。

In [3]:
print(app_train['TARGET'].value_counts(normalize=True))

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


**結果**：違約客戶僅佔約 8.07%，正常還款客戶佔 91.93%。這個比例確認了後續必須用 AUC / KS / Gini
而非 Accuracy 來評估模型，也確認了模型訓練時需要對不平衡做校正。

## 3. 缺失值分析

檢視各欄位缺失比例，找出最嚴重的前 20 個欄位。

In [4]:
missing = app_train.isnull().mean().sort_values(ascending=False)
print(missing.head(20))

COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_MEDI    0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_MODE       0.683550
FLOORSMIN_MODE              0.678486
FLOORSMIN_AVG               0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_AVG             0.664978
YEARS_BUILD_MODE            0.664978
YEARS_BUILD_MEDI            0.664978
OWN_CAR_AGE                 0.659908
LANDAREA_MEDI               0.593767
LANDAREA_AVG                0.593767
LANDAREA_MODE               0.593767
dtype: float64


**觀察**：缺失值比例最高的欄位幾乎全數集中在住房建物相關欄位（`COMMONAREA_*`、`LIVINGAPARTMENTS_*`、
`FLOORSMIN_*`、`YEARS_BUILD_*` 等），且同一概念常以 `_AVG`、`_MODE`、`_MEDI` 三種統計量重複出現，
彼此高度共線，不是三種獨立資訊。`OWN_CAR_AGE` 缺失比例高達 65.99%，需要進一步驗證是否為結構性缺失。

**處理策略（實作於 `src/features.py` 的 `clean_application()`）**：只保留 `_AVG` 版本，丟棄前先計算
「住房資訊缺失數量」作為獨立特徵（缺失本身可能帶有訊號）；類別型缺失獨立成 `'Missing'` 類別。

## 4. 驗證：`OWN_CAR_AGE` 缺失是否為結構性缺失

假設：`OWN_CAR_AGE` 缺失是因為客戶「沒有車」，而不是隨機遺失。用 `FLAG_OWN_CAR` 交叉檢查驗證這個假設。

In [5]:
car_check = pd.crosstab(app_train['FLAG_OWN_CAR'], app_train['OWN_CAR_AGE'].isnull())
print(car_check)

OWN_CAR_AGE    False   True 
FLAG_OWN_CAR                
N                  0  202924
Y             104582       5


**結果**：`FLAG_OWN_CAR == 'N'`（沒車）的 202,924 筆客戶，`OWN_CAR_AGE` **全部**缺失；有車的客戶裡只有
5 筆缺失。假設成立——這是結構性缺失，不是隨機遺失，因此不適合用中位數填補（會抹去「沒有車」這個訊號），
應填入哨兵值（如 0），並保留 `FLAG_OWN_CAR` 讓模型自行學習兩者的差異。

## 5. 異常值偵測：`DAYS_EMPLOYED` 的資料編碼錯誤

`DAYS_EMPLOYED`（已就業天數）正常應為負值（相對申請日的天數）。檢查其統計量與異常值分佈。

In [6]:
print(app_train['DAYS_EMPLOYED'].describe())
print('\n異常值(365243)筆數:', (app_train['DAYS_EMPLOYED'] == 365243).sum())
print('異常值佔比:', (app_train['DAYS_EMPLOYED'] == 365243).mean())

# 檢查異常值集中在哪個收入類型
print(app_train.loc[app_train['DAYS_EMPLOYED'] == 365243, 'NAME_INCOME_TYPE'].value_counts())

count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

異常值(365243)筆數: 55374
異常值佔比: 0.18007160719453938
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64


**結果**：`DAYS_EMPLOYED` 最大值為 365243（換算超過一千年），佔全體 18.01%（55,374 筆），且其中
55,352 筆對應 `Pensioner`（退休）、22 筆對應 `Unemployed`（失業）。這確認了這不是隨機遺失，而是資料庫
對「無業/退休、不適用」狀態的錯誤編碼——如果沒有抓到這點，會嚴重扭曲任何跟就業年資相關的統計量與模型學習。

**處理策略**：建立 `DAYS_EMPLOYED_ANOM` 旗標欄位記錄該異常，再將異常值替換為 `NaN`，避免污染後續分析。

## 6. 離群值檢查：`AMT_INCOME_TOTAL`

檢查收入欄位是否存在極端離群值。

In [7]:
print(app_train['AMT_INCOME_TOTAL'].describe())
print(app_train['AMT_INCOME_TOTAL'].sort_values(ascending=False).head(5))

count    3.075110e+05
mean     1.687979e+05
std      2.371231e+05
min      2.565000e+04
25%      1.125000e+05
50%      1.471500e+05
75%      2.025000e+05
max      1.170000e+08
Name: AMT_INCOME_TOTAL, dtype: float64
12840     117000000.0
203693     18000090.0
246858     13500000.0
77768       9000000.0
131127      6750000.0
Name: AMT_INCOME_TOTAL, dtype: float32


**結果**：最大值 1.17 億，與第二名（1800 萬）相差 6.5 倍，是典型的單筆離群值。其餘如 675 萬～1350 萬
雖然也偏高，但屬於合理的極端富有客戶範圍，不需特別處理。

**處理策略**：用 99.9 百分位數做截尾（clipping），而非直接刪除。信用風控場景中刪除離群列具有風險——
正式上線後仍可能出現真實的極端高收入客戶，模型應具備處理極端值的穩健性。

## 7. 套用正式清理邏輯

以上驗證過的處理邏輯已封裝進 `src/features.py` 的 `clean_application()`，套用後檢查結果是否符合預期。

In [8]:
from src.features import clean_application

app_train_clean = clean_application(app_train)
print(app_train_clean.shape)
print(app_train_clean[['AGE_YEARS', 'YEARS_EMPLOYED', 'DAYS_EMPLOYED_ANOM',
                        'HOUSING_INFO_MISSING_COUNT']].describe())

# 驗證 DAYS_EMPLOYED 異常值是否已被正確清除
print('清理後 YEARS_EMPLOYED 最大值:', app_train_clean['YEARS_EMPLOYED'].max())

(307511, 93)
           AGE_YEARS  YEARS_EMPLOYED  DAYS_EMPLOYED_ANOM  \
count  307511.000000   252137.000000       307511.000000   
mean       43.906898        6.527500            0.180072   
std        11.947950        6.402081            0.384248   
min        20.503765       -0.000000            0.000000   
25%        33.984943        2.099931            0.000000   
50%        43.121151        4.511978            0.000000   
75%        53.886379        8.692677            0.000000   
max        69.073235       49.040382            1.000000   

       HOUSING_INFO_MISSING_COUNT  
count               307511.000000  
mean                     8.182081  
std                      6.151516  
min                      0.000000  
25%                      0.000000  
50%                     10.000000  
75%                     14.000000  
max                     14.000000  
清理後 YEARS_EMPLOYED 最大值: 49.040382


**驗證通過**：欄位數從 122 降到 93（房屋建物三胞胎欄位已清除），`YEARS_EMPLOYED` 最大值回到合理的
幾十年範圍，確認 365243 的異常編碼已被成功處理。

## 8. 金融風控核心比率特徵

在原始欄位基礎上建構具明確金融意義的比率特徵：負債比、年金收入比、信貸商品比等，並聚合三個外部信用
評分來源（`EXT_SOURCE_1/2/3`）的統計量——這是本資料集中公認預測力最強的原始欄位組合。

In [9]:
from src.features import add_financial_ratios

app_train_fe = add_financial_ratios(app_train_clean)
print(app_train_fe.shape)

new_cols = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
            'CREDIT_GOODS_RATIO', 'INCOME_PER_PERSON', 'EXT_SOURCE_MEAN']
print(app_train_fe[new_cols].describe())

# 檢查有沒有除以零產生的 inf
print('inf 檢查:', np.isinf(app_train_fe[new_cols]).sum().sum())

(307511, 105)
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO    CREDIT_TERM  \
count        307511.000000         307499.000000  307499.000000   
mean              3.957948              0.180951      21.612320   
std               2.689355              0.094553       7.823823   
min               0.083333              0.007500       8.036674   
25%               2.018667              0.114800      15.614496   
50%               3.265067              0.162833      20.000000   
75%               5.159880              0.229080      27.099985   
max              84.736839              1.875965      45.305080   

       CREDIT_GOODS_RATIO  INCOME_PER_PERSON  EXT_SOURCE_MEAN  
count       307233.000000      307509.000000    307339.000000  
mean             1.122995       92654.648438         0.509251  
std              0.124045       67407.765625         0.149802  
min              0.150000        2812.500000         0.000006  
25%              1.000000       47250.000000         0.413648 

### 相關性方向驗證（Sanity Check）

理論上 `EXT_SOURCE_MEAN` 應與 `TARGET` 呈**負相關**（分數越高、違約機率越低）；`ANNUITY_INCOME_RATIO`、
`CREDIT_GOODS_RATIO` 等負債類比率應呈**正相關**（比率越高、違約機率越高）。這是驗證特徵工程有沒有做對
的重要習慣。

In [10]:
correlations = app_train_fe[new_cols + ['TARGET']].corr()['TARGET'].sort_values()
print(correlations)

EXT_SOURCE_MEAN        -0.222052
CREDIT_TERM            -0.032102
INCOME_PER_PERSON      -0.015310
CREDIT_INCOME_RATIO    -0.007745
ANNUITY_INCOME_RATIO    0.014239
CREDIT_GOODS_RATIO      0.069427
TARGET                  1.000000
Name: TARGET, dtype: float64


**結果**：`EXT_SOURCE_MEAN`（-0.222）方向正確且是目前最強的線性相關特徵；`ANNUITY_INCOME_RATIO`
（+0.014）、`CREDIT_GOODS_RATIO`（+0.069）方向也都正確。`CREDIT_INCOME_RATIO` 相關係數接近 0，
`CREDIT_TERM` 看似負相關但實際上有其業務解釋（能核准長期分期的客戶通常信用體質較好）——這也提醒了
一件事：**單看 Pearson 線性相關係數會低估這些特徵的真實預測力**，因為樹模型能抓到的是非線性交互作用，
這點在後續 SHAP 分析中會再次驗證與呼應。

## 9. 附屬表格整併：`bureau.csv`（過去信用局紀錄）

`bureau.csv` 記錄客戶在**其他銀行**過去/現在所有的信貸紀錄，一個客戶可能對應多筆信貸，需先用
`groupby` 聚合成「一個客戶一列」再併回主表。核心訊號涵蓋：信貸廣度（開過幾筆）、嚴重度（逾期天數）、
負債規模（尚欠總額）、額度使用率、以及信貸狀態分佈。

In [11]:
from src.features import aggregate_bureau, merge_bureau_features
import gc

bureau = pd.read_csv('../data/bureau.csv')
bureau = reduce_mem_usage(bureau)
print('bureau shape:', bureau.shape)
print('bureau 涵蓋幾個不重複客戶:', bureau['SK_ID_CURR'].nunique())

記憶體用量：271.48 MB → 168.36 MB (降低 38.0%)
bureau shape: (1716428, 17)
bureau 涵蓋幾個不重複客戶: 305811


In [12]:
bureau_agg = aggregate_bureau(bureau)
print('聚合後 shape:', bureau_agg.shape)

del bureau
gc.collect()

app_train_bureau = merge_bureau_features(app_train_fe, bureau_agg)
print('merge 後 shape:', app_train_bureau.shape)
print(app_train_bureau['BUREAU_HAS_RECORD'].value_counts(normalize=True))

聚合後 shape: (305811, 23)
merge 後 shape: (307511, 128)
BUREAU_HAS_RECORD
1    0.856851
0    0.143149
Name: proportion, dtype: float64


In [13]:
bureau_cols = [c for c in app_train_bureau.columns if c.startswith('BUREAU_')]
print(app_train_bureau[bureau_cols + ['TARGET']].corr()['TARGET'].sort_values())

BUREAU_STATUS_Closed                 -0.030812
BUREAU_HAS_RECORD                    -0.030789
BUREAU_AMT_CREDIT_SUM_MEAN           -0.019957
BUREAU_AMT_CREDIT_SUM_MAX            -0.019737
BUREAU_AMT_CREDIT_SUM_SUM            -0.014057
BUREAU_SK_ID_BUREAU_COUNT            -0.010020
BUREAU_AMT_CREDIT_SUM_DEBT_MEAN      -0.000637
BUREAU_STATUS_Bad debt                0.004003
BUREAU_CNT_CREDIT_PROLONG_SUM         0.004058
BUREAU_CREDIT_DAY_OVERDUE_MAX         0.005493
BUREAU_AMT_CREDIT_SUM_DEBT_SUM        0.007144
BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN    0.007150
BUREAU_CREDIT_DAY_OVERDUE_MEAN        0.008118
BUREAU_STATUS_Sold                    0.012058
BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM     0.013335
BUREAU_DAYS_CREDIT_ENDDATE_MIN        0.034281
BUREAU_DAYS_CREDIT_ENDDATE_MAX        0.036590
BUREAU_DAYS_CREDIT_ENDDATE_MEAN       0.046983
BUREAU_DAYS_CREDIT_MAX                0.049782
BUREAU_CREDIT_UTILIZATION             0.060235
BUREAU_STATUS_Active                  0.067128
BUREAU_DAYS_C

**關鍵發現**：`BUREAU_DAYS_CREDIT_MEAN`（+0.090）是最強的正相關特徵——客戶越常有近期新開信貸紀錄，
違約風險越高，符合信用風控的經典邏輯（近期頻繁申請新信貸本身就是財務壓力的訊號）。
`BUREAU_CREDIT_UTILIZATION`（+0.060）、`BUREAU_STATUS_Active`（+0.067）方向也都合理。
`BUREAU_AMT_CREDIT_SUM_MEAN` 呈負相關，需注意這是混雜因子（有完整信貸紀錄、額度較大的人通常是財力
穩健的族群），不宜直接解讀為「欠越多錢越安全」。

## 10. 附屬表格整併：`previous_application.csv`（本公司歷史申貸紀錄）

記錄客戶過去在 **Home Credit 自己** 申請貸款的歷史，核心訊號是「過去申請有沒有被拒絕過」，
這是本資料集公認第二強的特徵來源。

In [14]:
from src.features import aggregate_previous_application, merge_previous_application

prev = pd.read_csv('../data/previous_application.csv')
prev = reduce_mem_usage(prev)
print('prev shape:', prev.shape)

prev_agg = aggregate_previous_application(prev)
print('聚合後 shape:', prev_agg.shape)

del prev
gc.collect()

記憶體用量：671.81 MB → 525.27 MB (降低 21.8%)
prev shape: (1670214, 37)
聚合後 shape: (338857, 20)


0

In [15]:
app_train_full = merge_previous_application(app_train_bureau, prev_agg)
print('merge 後 shape:', app_train_full.shape)

prev_cols = [c for c in app_train_full.columns if c.startswith('PREV_')]
print(app_train_full[prev_cols + ['TARGET']].corr()['TARGET'].sort_values())

merge 後 shape: (307511, 148)
PREV_AMT_ANNUITY_MEAN       -0.034871
PREV_STATUS_Approved        -0.031553
PREV_AMT_ANNUITY_MAX        -0.028966
PREV_AMT_APPLICATION_MEAN   -0.021803
PREV_AMT_CREDIT_MEAN        -0.016114
PREV_AMT_APPLICATION_MAX    -0.012605
PREV_AMT_CREDIT_MAX         -0.008439
PREV_STATUS_Unused offer     0.000517
PREV_AMT_APPLICATION_SUM     0.004607
PREV_AMT_CREDIT_SUM          0.008308
PREV_DAYS_DECISION_MAX       0.016399
PREV_HAS_RECORD              0.018476
PREV_STATUS_Canceled         0.019031
PREV_SK_ID_PREV_COUNT        0.023513
PREV_CNT_PAYMENT_MEAN        0.027743
PREV_CNT_PAYMENT_MAX         0.029439
PREV_DAYS_DECISION_MEAN      0.046864
PREV_DAYS_DECISION_MIN       0.053434
PREV_STATUS_Refused          0.064469
PREV_REFUSAL_RATE            0.077671
TARGET                       1.000000
Name: TARGET, dtype: float64


**關鍵發現**：`PREV_REFUSAL_RATE`（+0.078）是該表最強的正相關特徵，驗證了假設——過去申請被拒的比例
越高，未來違約風險越高。`PREV_STATUS_Approved`（-0.032）方向也合理（核准越多次代表被銀行判定為
低風險的歷史紀錄越多）。

## 11. 其餘三張表（精簡處理）

`POS_CASH_balance.csv`、`installments_payments.csv`、`credit_card_balance.csv` 記錄的是分期還款的
月度明細，顆粒度較細，重要性次於前兩張表。核心訊號聚焦在還款行為的異常訊號（短繳、逾期天數、額度使用率、
ATM 提現行為）。三張表用類似的 `groupby` 聚合邏輯精簡處理，完整實作見 `src/features.py` 中的
`aggregate_installments()`、`aggregate_pos_cash()`、`aggregate_credit_card()`，此處不重複展開。

完整整併 5 張附屬表格後，最終訓練集規模為 **307,511 列 × 174 欄**（含 `TARGET`）。

## 12. EDA 結論與後續行動

本次探索性分析得出的關鍵洞察，直接對應到後續特徵工程與建模階段的設計決策：

1. **目標變數極度不平衡（8.07% 違約率）** → 後續模型訓練需用 `scale_pos_weight` / `class_weight` 校正，
   評估指標採用 AUC / KS / Gini 而非 Accuracy。
2. **住房建物欄位存在高度共線的三胞胎（`_AVG`/`_MODE`/`_MEDI`）** → 僅保留 `_AVG`，並將缺失數量轉為
   獨立特徵。
3. **`OWN_CAR_AGE`、`DAYS_EMPLOYED` 存在結構性缺失與編碼錯誤，而非隨機遺失** → 不能用中位數暴力填補，
   須先理解缺失/異常的成因，用旗標欄位保留訊號。
4. **`EXT_SOURCE_MEAN` 是目前最強的線性相關特徵**，但部分工程特徵（如 `CREDIT_TERM`）線性相關係數雖低，
   實際預測力可能透過非線性交互作用發揮，需留待 SHAP 分析進一步驗證。
5. **附屬表格（`bureau`、`previous_application`）的聚合特徵均驗證出符合業務直覺的相關性方向**，
   確認整併附表這件事對模型有實質貢獻，不是白做工。

下一步：將以上驗證過的邏輯正式收斂進 `src/build_features.py`，供 train / test 共用同一套 pipeline，
避免特徵不一致導致的模型失效風險，接著進入模型訓練與調校階段（見 `src/train_baseline.py`、
`src/tune_optuna.py`、`src/train_final.py`）。